In [1]:
!pip install xgboost imbalanced-learn

In [2]:
import pandas as pd

train = pd.read_csv("train.csv")
val = pd.read_csv("val.csv")
test = pd.read_csv("test.csv")

print(train.shape)
print(val.shape)
print(test.shape)

(4922, 31)
(1055, 31)
(1055, 31)


In [3]:
X_train = train.drop("Churn", axis=1)
y_train = train["Churn"]

X_val = val.drop("Churn", axis=1)
y_val = val["Churn"]

X_test = test.drop("Churn", axis=1)
y_test = test["Churn"]

In [4]:
from sklearn.linear_model import LogisticRegression

lr = LogisticRegression(
    max_iter=1000,
    class_weight="balanced"
)

lr.fit(X_train, y_train)

LogisticRegression(class_weight='balanced', max_iter=1000)

In [5]:
lr_pred = lr.predict(X_test)

lr_prob = lr.predict_proba(X_test)[:,1]

In [6]:
from sklearn.metrics import classification_report

print(classification_report(y_test, lr_pred))

              precision    recall  f1-score   support

           0       0.90      0.71      0.79       775
           1       0.49      0.78      0.60       280

    accuracy                           0.73      1055
   macro avg       0.69      0.74      0.70      1055
weighted avg       0.79      0.73      0.74      1055



In [7]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(
    random_state=42,
    class_weight="balanced"
)

rf.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', random_state=42)

In [8]:
rf_pred = rf.predict(X_test)

rf_prob = rf.predict_proba(X_test)[:,1]

In [9]:
print(classification_report(y_test, rf_pred))

              precision    recall  f1-score   support

           0       0.82      0.90      0.85       775
           1       0.61      0.44      0.51       280

    accuracy                           0.78      1055
   macro avg       0.71      0.67      0.68      1055
weighted avg       0.76      0.78      0.76      1055



In [10]:
from xgboost import XGBClassifier

xgb = XGBClassifier(
    random_state=42
)

xgb.fit(X_train, y_train)

XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=None, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric=None, feature_types=None,
              feature_weights=None, gamma=None, grow_policy=None,
              importance_type=None, interaction_constraints=None,
              learning_rate=None, max_bin=None, max_cat_threshold=None,
              max_cat_to_onehot=None, max_delta_step=None, max_depth=None,
              max_leaves=None, min_child_weight=None, missing=nan,
              monotone_constraints=None, multi_strategy=None, n_estimators=None,
              n_jobs=None, num_parallel_tree=None, ...)

In [11]:
xgb_pred = xgb.predict(X_test)

xgb_prob = xgb.predict_proba(X_test)[:,1]

In [12]:
print(classification_report(y_test, xgb_pred))

              precision    recall  f1-score   support

           0       0.83      0.87      0.85       775
           1       0.58      0.50      0.53       280

    accuracy                           0.77      1055
   macro avg       0.70      0.68      0.69      1055
weighted avg       0.76      0.77      0.76      1055



In [13]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'n_estimators':[100,200],
    'max_depth':[5,10,None]
}

grid_rf = GridSearchCV(
    RandomForestClassifier(
        class_weight="balanced",
        random_state=42
    ),
    param_grid,
    cv=5,
    scoring='f1'
)

grid_rf.fit(X_train, y_train)

print(grid_rf.best_params_)

{'max_depth': 10, 'n_estimators': 200}


In [14]:
param_grid = {
    'n_estimators':[100,200],
    'max_depth':[3,5],
    'learning_rate':[0.01,0.1]
}

grid_xgb = GridSearchCV(
    XGBClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='f1'
)

grid_xgb.fit(X_train, y_train)

print(grid_xgb.best_params_)

{'learning_rate': 0.1, 'max_depth': 3, 'n_estimators': 100}


In [15]:
results = pd.DataFrame({
    "Actual": y_test,
    "LR_Pred": lr_pred,
    "RF_Pred": rf_pred,
    "XGB_Pred": xgb_pred
})

results.to_csv("results.csv", index=False)

In [16]:
probs = pd.DataFrame({
    "LR_Prob": lr_prob,
    "RF_Prob": rf_prob,
    "XGB_Prob": xgb_prob
})

probs.to_csv("probabilities.csv", index=False)

In [17]:
importance = pd.DataFrame({
    "Feature": X_train.columns,
    "Importance": rf.feature_importances_
})

importance = importance.sort_values(
    by="Importance",
    ascending=False
)

print(importance.head(10))

                           Feature  Importance
8                     TotalCharges    0.176415
4                           tenure    0.167730
7                   MonthlyCharges    0.151037
26               Contract_Two year    0.048390
11     InternetService_Fiber optic    0.041466
28  PaymentMethod_Electronic check    0.037982
14              OnlineSecurity_Yes    0.027030
25               Contract_One year    0.026664
0                           gender    0.026308
6                 PaperlessBilling    0.026204


In [18]:
importance.to_csv(
    "feature_importance.csv",
    index=False
)

In [19]:
from google.colab import files

files.download("results.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [20]:
files.download("probabilities.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [21]:
files.download("feature_importance.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>